# DepthCrafter Video Depth on CUDA T4 (Colab)

Run [`generate_depth.py`](../generate_depth.py) with Tencent **DepthCrafter** on a T4.
Google Drive is used **only** for the input video and the output depth MP4.

**Runtime:** Runtime → Change runtime type → GPU (T4).

You need a Hugging Face token that can download
[`stabilityai/stable-video-diffusion-img2vid-xt`](https://huggingface.co/stabilityai/stable-video-diffusion-img2vid-xt)
(accept the license on that page first).

## 1. Clone the repo

In [ ]:
# Set your GitHub URL after pushing this repo, then run.
REPO_URL = "https://github.com/whizsid/dvd-video-depth-map.git"  # <-- edit me
REPO_DIR = "/content/dvd"

import os
from pathlib import Path

if Path(REPO_DIR).exists():
    print(f"Already cloned at {REPO_DIR}")
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
print("cwd:", os.getcwd())

## 2. Install pip and system dependencies

In [ ]:
# ffmpeg for OpenCV decode/encode; project deps.
# Colab ships CUDA torch — keep it. DepthCrafter is imported from vendor/ via sys.path.
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null
!pip install -q -r requirements.txt

print("Install done.")

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# @markdown Path relative to `MyDrive` (e.g. `depth/input.mp4`).
VIDEO_REL_PATH = "depth/input.mp4"  # @param {type:"string"}

DRIVE_ROOT = Path("/content/drive/MyDrive")
INPUT_VIDEO = DRIVE_ROOT / VIDEO_REL_PATH.strip().lstrip("/")
DRIVE_DIR = INPUT_VIDEO.parent

DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print("DRIVE_DIR:", DRIVE_DIR)
print("INPUT_VIDEO exists:", INPUT_VIDEO.exists(), "->", INPUT_VIDEO)

## 4. Check runtime

In [ ]:
import torch

assert torch.cuda.is_available(), "Enable a GPU runtime (Runtime → Change runtime type → T4)."
idx = torch.cuda.current_device()
name = torch.cuda.get_device_name(idx)
props = torch.cuda.get_device_properties(idx)
free, total = torch.cuda.mem_get_info(idx)
print(f"GPU: {name}")
print(f"VRAM: {props.total_memory / 1024**3:.1f} GiB total, "
      f"{free / 1024**3:.1f}/{total / 1024**3:.1f} GiB free")
print(f"torch {torch.__version__} | CUDA {torch.version.cuda}")
if "T4" not in name:
    print(f"Note: expected a T4; got {name!r}. Use --max-res 512 + CPU offload if VRAM is tight.")

## 5. Pull DepthCrafter + SVD-XT weights

In [ ]:
import os
from pathlib import Path

# Required for SVD-XT (accept license on HF first).
HF_TOKEN = os.environ.get("HF_TOKEN", "")  # or set: HF_TOKEN = "hf_..."
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
else:
    print("Warning: HF_TOKEN empty — SVD-XT download will fail without login/license.")

unet = Path("ckpt/DepthCrafter")
svd = Path("ckpt/stable-video-diffusion-img2vid-xt")

need = (not unet.exists()) or (not any(unet.iterdir()) if unet.exists() else True)
need = need or (not svd.exists()) or (not any(svd.iterdir()) if svd.exists() else True)
if need:
    print("Downloading DepthCrafter + SVD-XT into ./ckpt/ …")
    !python scripts/download_weights.py
else:
    print("Checkpoints already present:", unet.resolve(), svd.resolve())

assert unet.exists() and any(unet.iterdir()), f"Missing {unet}"
assert svd.exists() and any(svd.iterdir()), f"Missing {svd}"
print("Weights OK:", unet, "|", svd)

## 6. Small smoke test (optional)

Skip if you do not have a short local MP4. Prefer your Drive clip in the next cell.

In [ ]:
from pathlib import Path

# Use a short clip on local Colab disk if you have one; otherwise skip this cell.
SMOKE = Path("/content/smoke.mp4")
if not SMOKE.exists():
    print(f"No {SMOKE} — skip smoke test and run the Drive cell below.")
else:
    !python generate_depth.py \
      --input-video "{SMOKE}" \
      --output-dir outputs/smoke \
      --cache-dir .cache \
      --max-frames 40 \
      --max-res 512 \
      --window-size 40 \
      --overlap 10 \
      --cpu-offload sequential \
      --no-upsample
    outs = list(Path("outputs/smoke").glob("*_depth_gray.mp4"))
    print("Smoke outputs:", outs)
    assert outs, "Smoke test produced no depth video"

## 7. Run on Drive input → save output next to it

In [ ]:
from pathlib import Path
import shutil

assert INPUT_VIDEO.exists(), f"Put your video at {INPUT_VIDEO} (or edit VIDEO_REL_PATH above)."

# Copy off Drive onto local Colab disk — stabilize Farneback re-reads frames;
# Drive I/O makes that step look hung for tens of minutes.
LOCAL_VIDEO = Path("/content") / INPUT_VIDEO.name
if (not LOCAL_VIDEO.exists()) or LOCAL_VIDEO.stat().st_size != INPUT_VIDEO.stat().st_size:
    print(f"Copying {INPUT_VIDEO} -> {LOCAL_VIDEO} …")
    shutil.copy2(INPUT_VIDEO, LOCAL_VIDEO)
print("Local input:", LOCAL_VIDEO, "size_mb=", LOCAL_VIDEO.stat().st_size / 1e6)

# T4 defaults: max-res 512, sequential CPU offload, short window on low host RAM.
# generate_depth.py also auto-selects these when --cpu-offload auto.
!python generate_depth.py \
  --input-video "{LOCAL_VIDEO}" \
  --output-dir "{DRIVE_DIR}" \
  --cache-dir .cache \
  --max-res 512 \
  --cpu-offload sequential \
  --upsample-device cuda \
  --denoise-device cuda

stem = LOCAL_VIDEO.stem
out = DRIVE_DIR / f"{stem}_depth_gray.mp4"
print("Expected output:", out, "exists=", out.exists())
assert out.exists(), f"Missing output {out}"